# 02-Batch Summary, Filter, and Combine

**Purpose**: build a batch from repository fixtures, compare brief/full summaries, and practice state filtering and batch composition.

**Input**: representative Gaussian outputs under `tests/test_files/g16log/`.

**Expected conclusion**: a DataFrame contains frame IDs, status, and extended scientific fields; filters return new batches without mutating the original.


In [ ]:
from __future__ import annotations

from pathlib import Path
from molop.io import AutoParser


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not find repository root (pyproject.toml).")


repo_root = find_repo_root(Path.cwd())

## Batch Summary


In [ ]:
mix_batch = AutoParser((repo_root / "tests/test_files/mix_format/*").as_posix())
mix_batch

In [ ]:
mix_batch.to_summary_df()

In [ ]:
mix_batch.file_names

You can index a batch like a list to access a specific file.


In [ ]:
mix_batch[0].file_path

The index can also be a file path, which is more useful in large-scale scenarios where ordering is hard to obtain, for example:


In [ ]:
mix_batch[mix_batch[0].file_path].file_path

Quick inspection


In [ ]:
mix_batch.draw_grid_image(molsPerRow=2, subImgSize=(500, 500))

If the calculation results contain information that can be embedded into atoms or bonds, you can use `qm_embedded_rdmol` to get an embedded RDKit Mol. SDF/MOL files saved from such an object will contain that embedded information.


In [ ]:
mix_batch[0][-1].qm_embedded_rdmol()

## Filter Files


Enabling parallel processing can significantly improve throughput when handling many files.


In [ ]:
mix_batch.groupby(lambda x: x.pure_filename)

In [ ]:
mix_batch.groupby(lambda x: x.file_format)

In [ ]:
ts_batch = mix_batch.filter_state("ts")
ts_batch

In [ ]:
opt_batch = mix_batch.filter_state("opt")
opt_batch

In [ ]:
cation_batch = mix_batch.filter_value("charge", 1)
cation_batch


In [ ]:
anion_batch = mix_batch.filter_value("charge", 0, "<")
anion_batch


A built-in parallel executor is provided. You can run a custom function in parallel via `parallel_execute`, which returns a list when the function produces values. Side-effect-only functions that implicitly return `None` do not display a list of `None` values by default.


In [ ]:
mix_batch.parallel_execute(lambda x: print(x.charge))

## Combine


Simple boolean operations are supported, for example:


In [ ]:
ion_batch = cation_batch + anion_batch
ion_batch

In [ ]:
opt_batch_without_ts = opt_batch - ts_batch
opt_batch_without_ts